In [ ]:
from transformers import AutoModelForSemanticSegmentation, AutoProcessor
import requests


id2label = requests.get(
    "https://minio.lab.sspcloud.fr/projet-funathon/2026/project3/data/clcplus-backbone-id2label.json"
).json()
id2label = {int(k): v for k, v in id2label.items()}
label2id = {v: k for k, v in id2label.items()}


model_id = "nvidia/mit-b5"
model = AutoModelForSemanticSegmentation.from_pretrained(
    model_id,
    num_channels = 14,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

processor = AutoProcessor.from_pretrained(model_id)

[transformers] You passed `num_labels=12` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/1156 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b5
Key                                              | Status     |                                                                                                  
-------------------------------------------------+------------+--------------------------------------------------------------------------------------------------
classifier.bias                                  | UNEXPECTED |                                                                                                  
classifier.weight                                | UNEXPECTED |                                                                                                  
decode_head.batch_norm.running_var               | MISSING    |                                                                                                  
decode_head.batch_norm.bias                      | MISSING    |                                               

In [4]:
# Random placeholder loaders so the snippet runs standalone — replace with
# real Sentinel-2 tile loaders for actual training (see Exercise 4 / src/train.py).
import torch
from torch.utils.data import DataLoader

B, C, H, W = 4, 14, 64, 64
dummy_dataset = [
    {"pixel_values": torch.randn(C, H, W), "labels": torch.randint(0, 10, (H, W))}
    for _ in range(16)
]

def collate(batch):
    return {
        "pixel_values": torch.stack([x["pixel_values"] for x in batch]),
        "labels": torch.stack([x["labels"] for x in batch]),
    }

train_loader = DataLoader(dummy_dataset[:12], batch_size=B, collate_fn=collate)
val_loader   = DataLoader(dummy_dataset[12:], batch_size=B, collate_fn=collate)

# trainer = pl.Trainer(
#     max_epochs=20,
#     accelerator="auto",   # uses GPU if available, CPU otherwise
#     devices="auto",
#     callbacks=callbacks,
#     log_every_n_steps=10,
# )

# trainer.fit(module, train_loader, val_loader)

In [14]:
dummy_dataset[0]['labels'].shape

torch.Size([64, 64])

In [15]:
dummy_dataset[0]['pixel_values'].shape

torch.Size([14, 64, 64])

In [20]:
from torch.utils.data import Dataset

class SyntheticSegmentationDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        return {
            "pixel_values": item["pixel_values"].float(),
            "labels": item["labels"].long()
        }

train_dataset = SyntheticSegmentationDataset(dummy_dataset[:12])
val_dataset = SyntheticSegmentationDataset(dummy_dataset[12:])

In [23]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./segmentation-demo",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    save_strategy="no",  # Don't save, just demo
    logging_steps=1,
    do_train=True,
    do_eval=True,
)

def compute_metrics(eval_pred):
    """Optional: computes example accuracy."""
    logits, labels = eval_pred
    # logits: (batch, num_labels, H, W), labels: (batch, H, W)
    predictions = logits.argmax(1)
    acc = (predictions == labels).float().mean().item()
    return {"accuracy": acc}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_loader.dataset,
    eval_dataset=val_loader.dataset,
    compute_metrics=compute_metrics,  # optional but recommended
)

In [24]:
trainer.train()

/home/onyxia/work/funathon-project3/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,2.449710
2,2.458924
3,2.447127
4,2.444669
5,2.444060
6,2.439644
7,2.441479
8,2.440745
9,2.429877
10,2.428929


TrainOutput(global_step=15, training_loss=2.4355552991231284, metrics={'train_runtime': 72.5913, 'train_samples_per_second': 0.827, 'train_steps_per_second': 0.207, 'total_flos': 1747234089861120.0, 'train_loss': 2.4355552991231284, 'epoch': 5.0})

In [25]:
"""
End-to-end training entry point.

Steps:
  1. Seed RNGs (numpy / torch / cuda) so a training run is reproducible.
  2. Discover training/validation tiles and compute per-band normalisation
     statistics on the training split only — the same stats are reused for
     validation and at inference time (stored alongside the model in MLflow).
  3. Build Albumentations transforms (resize → flip → normalize → ToTensorV2).
  4. Hand the dataloaders to `pl.Trainer.fit`. Lightning drives the optimiser,
     scheduler, early-stopping and checkpointing.

Run from the project root with `python -m src.train` (or
`uv run python -m src.train`) so the `src.*` package imports resolve.
"""

import os
import random
import numpy as np
import torch
import pytorch_lightning as pl

from torch.utils.data import DataLoader, random_split
from torch import Generator
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

from src.data.loading import load_data
from src.data.normalization import compute_global_normalization
from src.data.transforms import build_transform
from src.training.lightning import get_lightning_module
from src.data.dataset import SegmentationDataset


# ==========================================================
# 1️⃣ CONFIG
# ==========================================================

CONFIG = {
    "train_regions": [
        "AT332",
        "BE100",
        "BE251",
        "BG322",
        "DEA54",
        "FRJ27",
        "LU000",
    ],
    "train_years": ["2018", "2021"],
    "test_regions": ["BE100", "DEA54", "LU000"],
    "test_year": "2021",
    "batch_size": 32,
    "test_batch_size": 16,
    "epochs": 20,
    "lr": 1e-3,
    "n_bands": 14,
    "resize": 512,
    "num_workers": os.cpu_count(),
    "seed": 42,
}


# ==========================================================
# 2️⃣ UTILS
# ==========================================================


def set_seed(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    pl.seed_everything(seed, workers=True)


def build_region_year_list(regions, years):
    return [f"{r}_{y}" for r in regions for y in years]


# ==========================================================
# 3️⃣ MAIN PIPELINE
# ==========================================================


# Reproducibilité
set_seed(CONFIG["seed"])

# -------- Dataset IDs --------
train_ids = build_region_year_list(
    CONFIG["train_regions"],
    CONFIG["train_years"],
)

test_ids = build_region_year_list(
    CONFIG["test_regions"],
    [CONFIG["test_year"]],
)

# -------- Normalisation --------
print("📊 Computing normalization...")
mean, std = compute_global_normalization(train_ids, CONFIG["n_bands"])

# -------- Chargement --------
print("📂 Loading data...")
train_patches, train_labels = load_data(train_ids)
test_patches, test_labels = load_data(test_ids)

# -------- Transforms --------
train_transform = build_transform(mean, std, augment=True, resize=CONFIG["resize"])

test_transform = build_transform(mean, std, augment=False, resize=CONFIG["resize"])

# -------- Dataset --------
full_dataset = SegmentationDataset(
    patchs=train_patches,
    labels=train_labels,
    n_bands=CONFIG["n_bands"],
    from_s3=False,
    transform=train_transform,
)

test_dataset = SegmentationDataset(
    patchs=test_patches,
    labels=test_labels,
    n_bands=CONFIG["n_bands"],
    from_s3=False,
    transform=test_transform,
)

# -------- Split train / val --------
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=Generator().manual_seed(CONFIG["seed"]),
)

# -------- DataLoaders --------
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    num_workers=CONFIG["num_workers"],
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG["test_batch_size"],
    num_workers=CONFIG["num_workers"],
    pin_memory=True,
)

Seed set to 42


📊 Computing normalization...
reading https://minio.lab.sspcloud.fr/projet-funathon/2026/project3/data/images/AT332/2021/filename2bbox.parquet



CalledProcessError: Command '['mc', 'cp', 'public/projet-funathon/2026/project3/data/images/AT332/2021/4405690_2665450_0_349.tif', 'data/data-preprocessed/patchs/AT332/2021/']' returned non-zero exit status 1.